# Task 2: BiDAF Reading Comprehension — Training & Evaluation

This notebook trains and evaluates two models on SQuAD v1.1:
1. **Baseline BiDAF** — GloVe embeddings + Character CNN
2. **BiDAF-BERT** — Frozen `bert-base-multilingual-uncased` embeddings

Run all cells in order. GPU runtime recommended (Runtime → Change runtime type → GPU).

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Upload Project Files

Upload these files from `project4/task2/` to the Colab working directory:
- `bidaf_model.py`
- `bidaf_bert_model.py`
- `data_utils.py`
- `train_evaluate.py`

Or clone your repo. The cell below checks they exist.

In [ ]:
import os
required = ['bidaf_model.py', 'bidaf_bert_model.py', 'data_utils.py', 'train_evaluate.py']
for f in required:
    assert os.path.exists(f), f'Missing {f} — please upload it'
print('All files present.')

## 2. Load SQuAD v1.1 Data

In [ ]:
from data_utils import load_squad

NUM_TRAIN = 5000
NUM_VAL = 1000

train_data, val_data = load_squad(num_train=NUM_TRAIN, num_val=NUM_VAL)
print(f'\nSample:\n  Q: {train_data[0]["question"]}\n  A: {train_data[0]["answer_text"]}')

## 3. Train Baseline BiDAF (GloVe + Char CNN)

In [ ]:
from train_evaluate import train_baseline

baseline_results = train_baseline(
    train_data, val_data, device,
    num_epochs=15,
    batch_size=32,
    lr=1e-3,
    hidden_dim=100,
)

print(f'\nBaseline Final — EM: {baseline_results["final_val_em"]:.2f}%, F1: {baseline_results["final_val_f1"]:.2f}%')

## 4. Train BiDAF-BERT (Frozen mBERT)

In [ ]:
from train_evaluate import train_bert_bidaf

bert_results = train_bert_bidaf(
    train_data, val_data, device,
    num_epochs=15,
    batch_size=16,
    lr=1e-3,
    hidden_dim=100,
)

print(f'\nBERT Final — EM: {bert_results["final_val_em"]:.2f}%, F1: {bert_results["final_val_f1"]:.2f}%')

## 5. Comparison

In [ ]:
import json

print(f'{"Metric":<25} {"Baseline BiDAF":>15} {"BiDAF-BERT":>15} {"Delta":>10}')
print('-' * 65)
print(f'{"Exact Match (%)":<25} {baseline_results["final_val_em"]:>15.2f} {bert_results["final_val_em"]:>15.2f} {bert_results["final_val_em"] - baseline_results["final_val_em"]:>+10.2f}')
print(f'{"F1 Score (%)":<25} {baseline_results["final_val_f1"]:>15.2f} {bert_results["final_val_f1"]:>15.2f} {bert_results["final_val_f1"] - baseline_results["final_val_f1"]:>+10.2f}')
print(f'{"Trainable Params":<25} {baseline_results["trainable_parameters"]:>15,} {bert_results["trainable_parameters"]:>15,}')
print(f'{"Total Params":<25} {baseline_results["total_parameters"]:>15,} {bert_results["total_parameters"]:>15,}')

# Save results
comparison = {
    'dataset': 'SQuAD v1.1',
    'train_size': NUM_TRAIN,
    'val_size': NUM_VAL,
    'device': str(device),
    'baseline_bidaf': baseline_results,
    'bert_bidaf': bert_results,
    'comparison': {
        'em_improvement': round(bert_results['final_val_em'] - baseline_results['final_val_em'], 2),
        'f1_improvement': round(bert_results['final_val_f1'] - baseline_results['final_val_f1'], 2),
    }
}

os.makedirs('results', exist_ok=True)
with open('results/task2_training_results.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print('\nResults saved to results/task2_training_results.json')

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

epochs_b = [h['epoch'] for h in baseline_results['training_history']]
epochs_bert = [h['epoch'] for h in bert_results['training_history']]

# Loss
axes[0].plot(epochs_b, [h['val_loss'] for h in baseline_results['training_history']], 'o-', label='Baseline')
axes[0].plot(epochs_bert, [h['val_loss'] for h in bert_results['training_history']], 's-', label='BERT')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Loss'); axes[0].set_title('Validation Loss')
axes[0].legend()

# EM
axes[1].plot(epochs_b, [h['val_em'] for h in baseline_results['training_history']], 'o-', label='Baseline')
axes[1].plot(epochs_bert, [h['val_em'] for h in bert_results['training_history']], 's-', label='BERT')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('EM (%)'); axes[1].set_title('Exact Match')
axes[1].legend()

# F1
axes[2].plot(epochs_b, [h['val_f1'] for h in baseline_results['training_history']], 'o-', label='Baseline')
axes[2].plot(epochs_bert, [h['val_f1'] for h in bert_results['training_history']], 's-', label='BERT')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1 (%)'); axes[2].set_title('F1 Score')
axes[2].legend()

plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 7. Download Results

Download the `results/` folder to include in your project submission.

In [ ]:
# For Colab: download results as zip
try:
    from google.colab import files
    import shutil
    shutil.make_archive('task2_results', 'zip', 'results')
    files.download('task2_results.zip')
except ImportError:
    print('Not running in Colab — results are in the results/ folder')